In [1]:
import os
import datetime as dt
import numpy as np
import glob
import json
import ens

import sys
sys.path.append('../run/')
import forecast_namelist as namelist
os.chdir('../src/') # back to original directory

debug = True

def myconverter(o):
    if isinstance(o, dt.datetime):
        return o.__str__()
        

def linkit(from_path,to_path ,link_option):
    '''from create_run_letkf'''
    
    if debug: print(f"LINKIT:  ARG[0]: {from_path}  ARG[1]:  {to_path }")

    if link_option == 1:
        LINK_CMD =f"ln -s {from_path} {to_path}" 
        
        if os.system(LINK_CMD) != 0:
            print(f"\n ERROR!!!\n ERROR!!! FAILIED TO EXECUTE: {LINK_CMD}\n ERROR!!!")
            sys.exit(1)
        print("Linked " + from_path + " to directory  " + to_path)
      
    if link_option == 2:
        CP_CMD = f"cp  {from_path} {to_path}" 
        
        if os.system(CP_CMD) != 0:
            print(f"\n ERROR!!!\n ERROR!!! FAILIED TO EXECUTE: {LINK_CMD}")
            if not os.path.exists(from_path):
                print(f"\nCOMMAND FAILED because {from_path} does not exist\n ")
            elif not os.path.join(to_path):
                print(f"COMMAND FAILED because{to_path} does not exist\n ")
            else:
                print("\nBOTH FILE AND DIRECTORY EXISTS....something else f__ked up here....\n")
            print("ERROR!!!\n") 
            sys.exit(1)
            
        print(f"Copied {from_path} to directory {to_path}")

    return

# directory that your existing experiment is in
base_dir             = namelist.base_dir
ncores               = namelist.ncores
forecast_freq        = namelist.forecast_output_freq
forecast_start       = namelist.forecast_start
forecast_end         = namelist.forecast_end
forecast_launch_freq = namelist.forecast_launch_freq # seconds. if more than zero, a new forecast will be launched at the time interval specified here
same_end_time        = namelist.same_end_time 

name = base_dir.split('/')[-1]
exp_name = f'{base_dir}/{name}.exp' # name of json file with all of the experiment info

# open experiment file to get experiment info
with open(exp_name, 'rb') as f: exper = json.load(f)

# determing forecast len
if forecast_launch_freq != 0:
    dtime = dt.timedelta(seconds=forecast_launch_freq)
    times = np.arange(forecast_start, forecast_end+ dtime, dtime)
else:
    times = [forecast_start]


In [2]:
for time in times:

    # determine length of each launched forecast
    if same_end_time:
        forecast_len = (forecast_end - time).seconds
    else:
        forecast_len = namelist.forecast_len


     # find the proper restart file to copy into each directory
    files, myDT = ens.FindRestartFiles(exper, time, ret_exp=False)


    # create a forecast_dir in base_dir
    fpath =  os.path.join(base_dir, dt.datetime.strftime(time, 'FORECAST_%Y%m%d_%H%M'))
    
    cmd   =f'mkdir {fpath}'
    os.system(cmd)

    # now copy in the cm1 files to the forecast directory
    for var in ['model', 'src', 'landsfc', 'namelist']:

        # first copy into forecast directory, link_option=2
        from_path = os.path.join(base_dir, os.path.basename(exper[var]))
        to_path   = os.path.join(fpath, os.path.basename(exper[var]))
        ret       = linkit(from_path, to_path, 2)


    # now set up member diirctories 
    fcst_members = []
    #for mem in range(1, exper['ne']+1, 1):
    for mem in range(1, 6, 1):

        # make directory
        mem_path = os.path.join(fpath, f'member{mem:03d}')
        fcst_members.append(mem_path)
        cmd = f'mkdir {mem_path}'
        os.system(cmd)

        # loop over cm1 files from forecast directory and link them to each member, link_option=1
        for var in ['model', 'src', 'landsfc', 'namelist']:
            from_path = os.path.join(fpath, os.path.basename(exper[var]))
            to_path   = mem_path
            ret       = linkit(from_path, to_path, 1)

        #now copy over the sounding file- NOTE: the "files" variable is sorted, so you can safely index this variable
        from_path = os.path.join(files[mem-1][:-17],'input_sounding')
        ret       = linkit(from_path, to_path, 2)

        # now copy the proper restart file over 
        from_path = files[mem-1]
        ret       = linkit(from_path, to_path, 2)
        

    # now, make a new experiment file to give to run_fcst.py  
    # clean up dictionary and update paths, leave DA_PARAMS because run_fcst.py expects them
    del exper['INIT']
    del exper['ADD_NOISE']
    
    exper['FORECAST'] = {'start':time, 'end':time+dt.timedelta(seconds=forecast_len), 'freq': forecast_freq}

    exper['base_dir']     = fpath
    exper['base_path']    = fpath
    exper['fcst_path']    = fpath
    exper['fcst_members'] = fcst_members
    
    exp_name = os.path.join(fpath,os.path.basename(fpath)+'.exp' )
    with open(exp_name, 'w') as handle:
        json.dump(exper, handle, default = myconverter)
    

    # YOU ARE NOW READY TO RUN THE FORECAST!!!!! 
    time_str = time.strftime('%Y,%m,%d,%H,%M,%S')

    os.system(f"python run_fcst.py -e {exp_name} --run_time {forecast_len} --freq {forecast_freq} -t {time_str} --ncores {ncores}")

    # make the full forecast netcdf
    os.system(f"python stats_calc.py -e {exp_name} -f --fcst_file") # combine forecast plots into single file


 ==> FindRestartFiles: Date and time supplied is 2024_05_08 20:40:00

 ==> FindRestartFile:  Found time 4200 in file:  /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/member001/cm1rst_000014.nc
LINKIT:  ARG[0]: /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/cm1.exe  ARG[1]:  /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/cm1.exe
Copied /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/cm1.exe to directory /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/cm1.exe
LINKIT:  ARG[0]: /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/onefile.F  ARG[1]:  /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/onefile.F
Copied /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/onefile.F to directory /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/onefile.F
LINKIT:  ARG[0]: /work/jessica.mcdonald/CM1_LETKF_2025/

 READ_NAMELIST: PROBLEM WITH NSSL_MP_PARAMS namelist: not found or bad token
 READ_NAMELIST: PROBLEM WITH NSSL_MP_PARAMS namelist: not found or bad token
 READ_NAMELIST: PROBLEM WITH NSSL_MP_PARAMS namelist: not found or bad token
 READ_NAMELIST: PROBLEM WITH NSSL_MP_PARAMS namelist: not found or bad token
 READ_NAMELIST: PROBLEM WITH NSSL_MP_PARAMS namelist: not found or bad token



 Executing command:  cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member001 ; mpirun -n 64 cm1.exe >> cm1.out 

 cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member001 ; mpirun -n 64 cm1.exe >> cm1.out completed....

 Executing command:  cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member002 ; mpirun -n 64 cm1.exe >> cm1.out 

 cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member002 ; mpirun -n 64 cm1.exe >> cm1.out completed....

 Executing command:  cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member003 ; mpirun -n 64 cm1.exe >> cm1.out 

 cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/member003 ; mpirun -n 64 cm1.exe >> cm1.out completed....

 Executing command:  cd /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_

Traceback (most recent call last):
  File "/work/jessica.mcdonald/CM1_LETKF_2025/src/stats_calc.py", line 271, in <module>
    timearr = np.arange(start_time, end_time+timedel, timedel)
TypeError: can only concatenate str (not "datetime.timedelta") to str


In [10]:
import xarray as xr
stats = xr.open_dataset(mem_path+'/cm1out_stats.nc')

/tmp/ipykernel_3890401/194414262.py:2: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  stats = xr.open_dataset(mem_path+'/cm1out_stats.nc')


In [12]:
exper

{'base_dir': '/work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040',
 'fprefix': 'cm1',
 'ne': 5,
 'model': 'cm1r21v1/run/cm1.exe',
 'src': 'cm1r21v1/run/onefile.F',
 'namelist': '/work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/namelist.input',
 'landsfc': 'cm1r21v1/run/LANDUSE.TBL',
 'sounding': 'soundings/',
 'radar_obs': '/work/jessica.mcdonald/CM1_LETKF_2025/run/observations/8may24_cm1_5km_obs.csv',
 'YEAR': 2024,
 'MONTH': 5,
 'DAY': 8,
 'HOUR': 19,
 'MINUTE': 30,
 'SECOND': 0,
 'lat0': 35.23583,
 'lon0': -97.46194,
 'hgt': 0,
 'xoffset': -180000.0,
 'yoffset': -180000.0,
 'microphysics': 'zvdh',
 'cook_period': 1800,
 'cook_freq': 300,
 'forecast_length': 3600,
 'forecast_freq': 300,
 'DA_PARAMS': {'obs_errors': {'11': ['VR', 3.0],
   '12': ['DBZ', 7.0],
   '13': ['DBZ0', 5.0],
   '14': ['DBZ0_W', 0.5]},
  'aInflate': 1,
  'outlier': 3,
  'nthreads': 8,
  'assim_window': 15,
  'assim_freq': 900,
  'async_freq': 0,
  'cook': 1800,
  'addit

In [37]:
timedel =dt.timedelta(seconds = exper['forecast_freq'])
da_end = dt.datetime.strptime(exper['DA_PARAMS']['DA_end_time'], '%Y-%m-%d %H:%M:%S') 
start_time = da_end + timedel
end_time = da_end + dt.timedelta(seconds = exper['forecast_length']) 

In [4]:
f"python stats_calc.py -e {exp_name} -f --fcst_file"

'python stats_calc.py -e /work/jessica.mcdonald/CM1_LETKF_2025/experiments/fcst_test/FORECAST_20240508_2040/FORECAST_20240508_2040.exp -f --fcst_file'

In [12]:
exper['FORECAST']['start']

datetime.datetime(2024, 5, 8, 20, 40)